# Model Results Viewer

This notebook loads and displays all evaluation results from a trained model's results directory.

**Usage**: Set the `RESULTS_DIR` variable below to the path of the results folder (e.g., `results_modality_resnet18`, `results_baseline_vit`, etc.) and run all cells.

In [ ]:
# --- Configuration ---
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

# Set this to your results directory (e.g., 'results_modality_resnet18')
RESULTS_DIR = Path("results_modality_resnet18")   # <--- CHANGE THIS

# Check if directory exists
if not RESULTS_DIR.exists():
    raise FileNotFoundError(f"Results directory not found: {RESULTS_DIR}")

print(f"Loading results from: {RESULTS_DIR.resolve()}")

## 1. Validation Results

In [ ]:
# Load validation CSV files
val_overall = pd.read_csv(RESULTS_DIR / "val_overall.csv")
val_per_class = pd.read_csv(RESULTS_DIR / "val_per_class.csv")
val_per_fst = pd.read_csv(RESULTS_DIR / "val_per_fst.csv")

display(Markdown("### Overall Metrics"))
display(val_overall.style.hide(axis='index').set_caption("Validation Overall"))

display(Markdown("### Per‑Class Metrics"))
display(val_per_class.style.hide(axis='index').set_caption("Validation Per Class"))

display(Markdown("### Per‑FST Accuracy"))
display(val_per_fst.style.hide(axis='index').set_caption("Validation Per FST"))

In [ ]:
# Show plots for validation
print("Confusion Matrix")
display(Image(filename=RESULTS_DIR / "val_confusion.png"))

print("Per‑Class Metrics")
display(Image(filename=RESULTS_DIR / "val_per_class.png"))

print("Fairness Metrics")
display(Image(filename=RESULTS_DIR / "val_fairness.png"))

## 2. Test Results (if available)

In [ ]:
test_overall_path = RESULTS_DIR / "test_overall.csv"
if test_overall_path.exists():
    test_overall = pd.read_csv(test_overall_path)
    test_per_class = pd.read_csv(RESULTS_DIR / "test_per_class.csv")
    test_per_fst = pd.read_csv(RESULTS_DIR / "test_per_fst.csv")
    
    display(Markdown("### Overall Metrics"))
    display(test_overall.style.hide(axis='index').set_caption("Test Overall"))
    
    display(Markdown("### Per‑Class Metrics"))
    display(test_per_class.style.hide(axis='index').set_caption("Test Per Class"))
    
    display(Markdown("### Per‑FST Accuracy"))
    display(test_per_fst.style.hide(axis='index').set_caption("Test Per FST"))
    
    print("Confusion Matrix")
    display(Image(filename=RESULTS_DIR / "test_confusion.png"))
    
    print("Per‑Class Metrics")
    display(Image(filename=RESULTS_DIR / "test_per_class.png"))
    
    print("Fairness Metrics")
    display(Image(filename=RESULTS_DIR / "test_fairness.png"))
else:
    print("No test results found.")

## 3. Cross‑Dataset Evaluation (Derm7pt)

In [ ]:
cross_summary_path = RESULTS_DIR / "cross_dataset_summary.csv"
if cross_summary_path.exists():
    cross_summary = pd.read_csv(cross_summary_path)
    display(Markdown("### Cross‑Dataset Summary (Derm7pt)"))
    display(cross_summary.style.set_caption("Derm7pt Results"))
else:
    print("No cross‑dataset summary found.")

# Also display individual cross‑dataset per‑split results if available
cross_files = sorted(RESULTS_DIR.glob("cross_*_overall.csv"))
for f in cross_files:
    split_name = f.stem.replace("_overall", "")
    df = pd.read_csv(f)
    display(Markdown(f"#### {split_name}"))
    display(df.style.hide(axis='index'))
    
    # Show corresponding plots
    conf_path = RESULTS_DIR / f"{split_name}_confusion.png"
    if conf_path.exists():
        display(Image(filename=conf_path))
    
    per_class_path = RESULTS_DIR / f"{split_name}_per_class.png"
    if per_class_path.exists():
        display(Image(filename=per_class_path))
    
    fair_path = RESULTS_DIR / f"{split_name}_fairness.png"
    if fair_path.exists():
        display(Image(filename=fair_path))

## 4. Training History & t‑SNE

In [ ]:
# Training curves
train_curve_path = RESULTS_DIR / "training_curves.png"
if train_curve_path.exists():
    print("Training Curves")
    display(Image(filename=train_curve_path))
else:
    print("Training curves not found.")

# t‑SNE visualization
tsne_path = RESULTS_DIR / "tsne_test.png"
if tsne_path.exists():
    print("t‑SNE of Test Set Embeddings")
    display(Image(filename=tsne_path))
else:
    print("t‑SNE plot not found.")

## 5. Additional Info (History JSON)

In [ ]:
history_path = RESULTS_DIR / "history.json"
if history_path.exists():
    with open(history_path, "r") as f:
        history = json.load(f)
    # Show last few epochs and best values
    df_history = pd.DataFrame(history)
    display(Markdown("### Training History (Last 5 Epochs)"))
    display(df_history.tail(5).style.format("{:.4f}"))
    
    # Best values
    best_auroc = max(df_history["val_auroc"].dropna())
    best_f1 = max(df_history["val_macro_f1"].dropna())
    print(f"Best Validation AUROC: {best_auroc:.4f}")
    print(f"Best Validation Macro-F1: {best_f1:.4f}")
else:
    print("history.json not found.")